In [1]:
import matplotlib.pyplot as plt
from PIL import Image
from brokenaxes import BrokenAxes
from intervaltree import Interval, IntervalTree
import numpy as np
import os
import random
import matplotlib.cm as cm
import argparse
import pandas as pd
import subprocess
import shutil

def merge_intervals(intervals):
    """Merge overlapping x-axis intervals using IntervalTree."""
    tree = IntervalTree(Interval(start, end) for start, end in intervals)
    #merge overlapping intervals
    tree.merge_overlaps() 
    #return list (not tuple) 
    return sorted((iv.begin, iv.end) for iv in tree)

# function to plot each dataset
def plot_dataset(dataset, name, output_folder):
    num_blocks = len(dataset)
    cmap = cm.get_cmap('tab20', num_blocks)
    fig, ax = plt.subplots(figsize=(12, 4))
    for block_index, block in enumerate(dataset):
        x_min = min(int(item[0]) for item in block)
        x_max = max(int(item[1]) for item in block)
        #print(f"Dataset {dataset_idx}: xlim = ({x_min}, {x_max})")
        parameter = 1
        # Step 1: Create unique x-limits based on dataset
        important_ranges = []
        important_ranges = [(start, end) for start, end, _ in block]
        merged_xlims = merge_intervals(important_ranges)
        print(merged_xlims)
        # Step 3: Plot horizontal lines
        distance = 0
        color = cmap(block_index)
        for start, end, line_count in block:
            print(start,end,line_count)
            if merged_xlims and int(start) == merged_xlims[0][0]:
                # logging.info("start")
                # logging.info(start)
                distance = 0
                merged_xlims.pop(0)
            y_positions = [distance + parameter * (i + 1) for i in range(int(line_count))]
            print(y_positions)
            # random_color = np.random.rand(3,)
            for y in y_positions:
                plt.hlines(y, start, end, colors=color, linestyles='solid', linewidth=1)
            distance = y_positions[-1]
            

    ax.set_xlabel("Genome Position")
    ax.set_ylabel("Coverage")    
    # plt.show()

    #save the plot to a buffer as a grayscale image
    buffer_path = f"{output_folder}/simulation_1_200_gaps_{name}_blocks.png"
    plt.savefig(buffer_path, format='png', dpi=300, bbox_inches='tight')
    plt.close()

def collect_all_blocks_data(simulation_data_folder, files):
    bed_blocks = {}
    block_lines = []
    # print(files)
    for file_name in files:
        try:
            file_path = os.path.join(simulation_data_folder, file_name)
            with open(file_path, "r") as file:
                bed_blocks[file_name] = []
                chrom_blocks = []
                line_block = []
                collecting = False
                for line in file:
                    line = line.strip()
                    if line == "":
                        collecting = False
                        if line_block:
                            bed_blocks[file_name].append(line_block)
                            line_block = []
                    elif line == "original bed lines:":
                        collecting = True
                    elif collecting:
                        columns = line.split('\t')
                        line_block.append([columns[0],columns[1],columns[2],columns[3], columns[4], columns[5], columns[6]])
        except Exception as e:
            print(f"Error opening file '{file_path.get(file_name, 'UNKNOWN')}': {e}")
    return bed_blocks

def cluster_blocks_by_chrom(bed_blocks, files):
    group_all = {}
    for file_name in files:
        group = {}
        # group[chrom] = []
        group_all[file_name] = {}
        for block in bed_blocks[file_name]:
            chrom = block[0][0]
            if chrom not in group:
                group[chrom] = []
            group[chrom].append(block)
        group_all[file_name] = group
    return group_all

def simulate_data_generation(original_data, blocks_number, start_point, file_name):
    # print("start original data")
    # print(original_data)
    # print("end original data")

    # print("file_name")
    # print(file_name)
    file_id = 0
    if "symmetric" in file_name:
        file_id = 1
    elif "fully_overlap" in file_name:
        file_id = 2
    elif "partial" in file_name:
        file_id = 3
    elif "singleton" in file_name:
        file_id = 4


    random_block_id = random.randint(0, blocks_number - 1)
    original_min_start_point = min(int(line[1]) for line in original_data[random_block_id])
    original_max_end_point = max(int(line[2]) for line in original_data[random_block_id])
    move_units = original_min_start_point - start_point
    max_end_point = original_max_end_point - move_units
    whole_block_data = []
    block_data = []

    for line in original_data[random_block_id]:
        line_start = int(line[1])
        line_end = int(line[2])
        line_count = int(line[6])
        # print(line[1],line[2],line[3],min_start_point)
        count = 50 if line_count > 50 else line_count
        # print(line_start - min_start_point, line_end - min_start_point, count)
        whole_block_data.append([line[0], line_start - move_units, line_end - move_units, line[3], line[4], line[5], count, file_id])
        block_data.append([line_start - move_units, line_end - move_units, count])
    # print("aaa")
    return block_data, whole_block_data, max_end_point

def generate_all_data(bed_blocks_group_by_chrom, files, blocks_number, smallest_gap, biggest_gap):
    number = 0
    dataset = []
    whole_dataset = {}
    random_file = random.choice(files)
    random_chrom = random.choice(list(bed_blocks_group_by_chrom[random_file].keys()))
    start_point = 1 
    while number < blocks_number:
        random_file = random.choice(files)
        actual_blocks_number = len(bed_blocks_group_by_chrom[random_file][random_chrom])
        block_data, whole_block_data, end_point = simulate_data_generation(bed_blocks_group_by_chrom[random_file][random_chrom], actual_blocks_number, start_point, random_file)
        dataset.append(block_data)
        whole_dataset[number] = whole_block_data
        # print("whole_block_data", whole_block_data)
        start_point = end_point + random.randint(smallest_gap, biggest_gap)
        number += 1
    return dataset,whole_dataset

def create_bed_file(result, bed_file_path):
    data = []
    for value_lists in result.values():
        # print("value_lists",value_lists)
        # for sublist in value_lists:
        data.extend(value_lists)
    # print("data:",data)
    df = pd.DataFrame(data, columns=['chrome', 'start', 'end','sequence','id','same_ids','same_seq_count','file_id'])
    # df['chrome'] = "NZ_LR134354.1"
    df.sort_values(by=['chrome', 'start', 'end'], ascending=[True, True, True], inplace=True)
    sorted_data = []
    sorted_data = df.values.tolist()

    # write data to the bed file
    with open(bed_file_path, "w") as file:
        for line in sorted_data:
            file.write("\t".join(map(str, line[:-1])) + "\n")

def clean_folder(folder_path):
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)  # remove file or symlink
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)  # remove folder and all its contents
        except Exception as e:
            print(f'Failed to delete {file_path}. Reason: {e}')

def run_simulation_code(code_folder, bed_file_path, output_folder):
    ##run additonal step
    clean_folder(os.path.join(output_folder,"results"))
    script_path = os.path.join(code_folder, "classify_mapping_patterns.py")

    command = [
        "python3", script_path,
        "-input_bedfile", bed_file_path,
        "-output_folder", os.path.join(output_folder,"results")
    ]

    subprocess.run(command, check=True)

def normalize_id(s):
    return '|'.join(sorted(s.split('|')))

# def new_simulation_redecide_class(output_folder):
#     df_new_blocks = pd.read_csv(os.path.join(output_folder, "results","final_all_blocks_table.csv"),sep = ",")
#     df_new_blocks['normalized_id'] = df_new_blocks['representative_SeqID'].apply(normalize_id)
#     df['combined_label'] = ''
#     #compare every row
#     for i in range(len(df)):
#         id_i = df.loc[i, 'normalized_id']
#         combined =[]
#         combined.append(df.loc[i, 'dataset_type'])

#         for j in range(len(df)):
#             if i == j:
#                 continue
#             id_j = df.loc[j, 'normalized_id']
#             if id_i.issubset(id_j):
#                 combined.append(df.loc[j, 'dataset_type'])

#         if combined:
#             df.at[i, 'combined_label'] = ','.join(combined)
def get_df_old_blocks(result):
   #['chrome', 'start', 'end','sequence','id','same_ids','same_seq_count','file_id'])
    data =[]
    for key, value in result.items():
        SeqID = []
        data_type = 0
        for inner_list in value:
            SeqID.append(inner_list[4])
            data_type = int(inner_list[-1])
        representative_SeqID = "|".join(SeqID) 
        data.append([representative_SeqID,data_type])
    df = pd.DataFrame(data, columns=['representative_SeqID','dataset_type'])
    return df
        # for sublist in value_lists:
        # data.extend(value_lists)
    # print("data:",data)

def compare_original_and_simulation(output_folder, result):
    # df_old_blocks = pd.read_csv(os.path.join(output_folder, "final_all_blocks_table.csv"),sep = ",")
    # print(df_old_blocks)
    df_old_blocks = get_df_old_blocks(result)
    df_new_blocks = pd.read_csv(os.path.join(output_folder, "results","final_all_blocks_table.csv"),sep = ",")
    print(df_new_blocks)
   
    df_new_blocks['normalized_id'] = df_new_blocks['representative_SeqID'].apply(normalize_id)
    df_old_blocks['normalized_id'] = df_old_blocks['representative_SeqID'].apply(normalize_id)

    merged = df_old_blocks.merge(df_new_blocks[['normalized_id']], on='normalized_id', how='left', indicator=True)
    merged['match'] = (merged['_merge'] == 'both').astype(int)
    correct_number = {}
    wrong_number = {} 
    new_correct_number = {}
    type_list = [1,2,3,4]
    print("merged")
    print(merged)
    for real_type in type_list:
        correct_number[real_type] = ((merged['match'] == 1) & (merged["dataset_type"] == real_type)).sum()
        wrong_number[real_type] = ((merged['match'] == 0) & (merged["dataset_type"] == real_type)).sum()
    
    for real_type in type_list:
        new_correct_number[real_type] = ((df_new_blocks["dataset_type"] == real_type)).sum()
    print("correct_number, new_correct_number, wrong_number")
    print(correct_number, new_correct_number, wrong_number)
    return correct_number, new_correct_number, wrong_number

def calculate_precision_recall(correct_number, new_correct_number, wrong_number):
    type_list = [1,2,3,4]
    precision = {}
    recall = {}
    for real_type in type_list:
        tp = correct_number[real_type]
        fp = new_correct_number[real_type] - correct_number[real_type]
        fn = wrong_number[real_type]
        precision[real_type] = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall[real_type] = tp / (tp + fn) if (tp + fn) > 0 else 0
    return precision, recall
# def produce_new_simualation_data(output_folder):
#     produce_new_simualation_data(output_folder)


In [2]:
def simulate_blocks(input_files, number_of_blocks, block_gap_min_threshold, block_gap_max_threshold, output_folder, code_folder, run_number):
    # simulation_data_folder = "simulation_data"
    # files = ["fully_symmetric_blocks.txt", "fully_overlapping_blocks.txt", 
    #             "partially_overlapping_blocks.txt", "singleton_blocks.txt", "other_situation_blocks.txt"]
    #prepare all the data
    input_files_list = input_files.split()
    if not input_files_list:
        raise ValueError(f"input files' paths: '{input_files}' is wrong")
    files = []
    for file_path in input_files_list:
        files.append(os.path.basename(file_path))
    simulation_data_folder = os.path.dirname(input_files_list[0])

    bed_blocks = collect_all_blocks_data(simulation_data_folder, files)
    # print(bed_blocks[files[0]])
    bed_blocks_group_by_chrom = cluster_blocks_by_chrom(bed_blocks, files)
    # print(bed_blocks_group_by_chrom)
    # print(bed_blocks_group_by_chrom[files[0]])
    #start simulating data
    original_simulation_dataset, whole_dataset = generate_all_data(bed_blocks_group_by_chrom, files, int(number_of_blocks), int(block_gap_min_threshold), int(block_gap_max_threshold))
    # print("simulation_dataset start:")
    # print(original_simulation_dataset)
    # print("simulation_dataset end.")
    print("whole_dataset start:")
    print(whole_dataset)
    print("whole_dataset end.")
    #plot_dataset(simulation_dataset, number_of_blocks, output_folder)
    ###create simulation data to bed file
    bed_file_path = os.path.join(output_folder,"sorted_representative_sequence_results_original.bed")
    create_bed_file(whole_dataset, bed_file_path)
    run_simulation_code(code_folder, bed_file_path, output_folder)
    # new_simulation_df = new_simulation_redecide_class(output_folder)
    correct_number, new_correct_number, wrong_number = compare_original_and_simulation(output_folder, whole_dataset)
    precision, recall = calculate_precision_recall(correct_number,new_correct_number, wrong_number)
    print("precision and recall")
    print(precision, recall)
    return precision,recall
    # plot_dataset(original_simulation_dataset, "old", output_folder, )
    # new_simulation_dataset = produce_new_simualation_data(output_folder)
    # plot_dataset(simulation_dataset, "new", output_folder, )

In [3]:
simulation_data_folder = "simulation_data"
code_folder ="../"
files = ["simulation_data/fully_symmetric_blocks.txt", "simulation_data/fully_overlapping_blocks.txt", 
             "simulation_data/partially_overlapping_blocks.txt", "simulation_data/singleton_blocks.txt"]
precision_single, recall_single = simulate_blocks(f"{files[0]} {files[1]} {files[2]} {files[3]}", 3, 1, 200, simulation_data_folder, code_folder, 1)
# precision = []
# recall = []
# for i in range(10):
#     precision_single, recall_single = simulate_blocks(f"{files[0]} {files[1]} {files[2]} {files[3]}", 15, 1, 200, simulation_data_folder, code_folder, i)
#     precision.append(precision_single)
#     recall.append(recall_single)


    


whole_dataset start:
{0: [['NC_002516.2', 1, 28, 'GCCAGCGGGTCGTTGAAGGTCTTGTTCA', '2711597', '2711597|4344017', 2, 2], ['NC_002516.2', 2, 31, 'GAACAAGACCTTCAACGACCCGCTGGCTGC', '2073157', '2073157', 1, 2], ['NC_002516.2', 2, 31, 'GAACAAGACCTTCAACGACCCGCTGGCGGC', '2900184', '2900184|10844924|15723820|20360631|25169945', 5, 2], ['NC_002516.2', 3, 36, 'AACAAGACCTTCAACGACCCGCTGGCGGCCAAGG', '24410067', '24410067', 1, 2], ['NC_002516.2', 6, 36, 'AAGACCTTCAACGACCCGCTGGCGGCCAAGG', '6159538', '6159538', 1, 2], ['NC_002516.2', 20, 36, 'TCCCGGTGGCGGCCAAGG', '1252644', '1252644', 1, 2], ['NC_002516.2', 22, 39, 'CGCTGGCGGCCAAGGTGC', '1832823', '1832823|8816053|13928559|22271885|27529499', 5, 2]], 1: [['NC_002516.2', 170, 193, 'GCAGCTTCTACGCGGCGGTCCGGC', '10408466', '10408466|12940656|20056432', 3, 4]], 2: [['NC_002516.2', 260, 291, 'TGATCGCCGCCGCCGTTTCCTTCCGTCGAGCC', '6528184', '6528184', 1, 3], ['NC_002516.2', 275, 295, 'GCGTTGATCGCCGCCGCCGTT', '4213492', '4213492|14885145|16267401', 3, 3], ['NC_002